In [ ]:
from pyspark.sql.functions import max

# Read the newly processed incremental data from Bronze
df_bronze = spark.table("LH_Nilay.bronze.bronze_orders")

# Extract the maximum LastModifiedDate from the current batch
max_watermark_row = df_bronze.select(max("LastModifiedDate").alias("MaxDate")).collect()
max_watermark_val = max_watermark_row[0]["MaxDate"]

if max_watermark_val:
    # Update the watermark table with the new max date
    spark.sql(f"""
        UPDATE LH_Nilay.meta.watermark_table
        SET WatermarkValue = cast('{max_watermark_val}' as timestamp)
        WHERE TableName = 'Orders'
    """)
    print(f"Watermark updated successfully to: {max_watermark_val}")
else:
    print("No new data found to update the watermark.")